In [ ]:
# --- CONSTANTS AND ENVIRONMENT SETUP ---

# 3x3 Grid (Row, Col) where (0, 0) is the top-left corner.
# The user's coordinates (r, c) map to (r-1, c-1) in this 0-indexed model.
# Grid Map (user coordinates):
# (1,3) BLOCKED | (2,3) | (3,3) GOAL/DIAMOND
# (1,2)         | (2,2) | (3,2)
# (1,1) START   | (2,1) | (3,1) FIRE
GRID_ROWS = 3
GRID_COLS = 3

# Define Special States (0-indexed)
# User's (1, 1) -> (0, 0)
START_STATE = (0, 0)
# User's (3, 3) Diamond Goal -> (2, 2)
GOAL_STATE = (2, 2)
# User's (3, 1) Fire -> (2, 0)
FIRE_STATE = (2, 0)
# User's (1, 3) Block -> (0, 2)
BLOCKED_STATE = (0, 2)

# Set of possible actions
ACTIONS = ['UP', 'DOWN', 'LEFT', 'RIGHT']

# MDP Parameters
GAMMA = 0.9      # Discount factor (Future rewards are worth 90% of current rewards)
CONVERGENCE_THRESHOLD = 0.0001 # Max change in utility before stopping iteration

# --- REWARD POLICY R(s) ---
# A function to define the immediate reward for being in a state.
# Action costs are incorporated by applying a small negative reward for non-terminal states.
def REWARD(state):
    (r, c) = state

    # R(s) for Terminal States
    if state == GOAL_STATE:
        return 10.0  # High positive reward for reaching the Diamond Goal (Prize)

    if state == FIRE_STATE:
        return -100.0 # High negative penalty for hitting Fire (Penalty)

    # Blocked state is special in that you can't enter it, so its R(s) is typically 0
    if state == BLOCKED_STATE:
        return 0.0

    # R(s) for Non-Terminal States
    # Cost for every action taken in a non-terminal state (Action Cost)
    return -0.04

# --- TRANSITION MODEL T(s' | s, a) ---
# This problem implies deterministic movement: T(s' | s, a) = 1 or 0.
def TRANSITION_MODEL(state, action):
    (r, c) = state

    # Terminal states (Goal and Fire) remain terminal in the utility calculation
    if state == GOAL_STATE or state == FIRE_STATE:
        return state

    # Calculate the intended next state
    if action == 'UP':
        r_next, c_next = r - 1, c
    elif action == 'DOWN':
        r_next, c_next = r + 1, c
    elif action == 'LEFT':
        r_next, c_next = r, c - 1
    elif action == 'RIGHT':
        r_next, c_next = r, c + 1
    else:
        # Should not happen
        return state

    # 1. Check for boundaries (The 'walls' constraint)
    if not (0 <= r_next < GRID_ROWS and 0 <= c_next < GRID_COLS):
        # Agent bumps into a wall, remains in the current state
        return state

    # 2. Check for Blocked State Constraint
    if (r_next, c_next) == BLOCKED_STATE:
        # Agent attempts to move into a blocked state, remains in current state
        # This handles the policy calculation when a blocked state is encountered.
        return state

    # Successful deterministic transition
    return (r_next, c_next)

# --- VALUE ITERATION ALGORITHM (Solution of MDP) ---

# The algorithm calculates the optimal utility of every state U(s)
# and the optimal policy PI(s) derived from it.
def CALCULATE_OPTIMAL_POLICY():
    # 1. Initialize Utility U(s) for all states
    # U is a 2D list/array to store utility values
    U = [[0.0 for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]

    # PI is a 2D list/array to store the optimal action (policy)
    PI = [['' for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]

    while True:
        U_prev = [row[:] for row in U] # Deep copy of U
        max_change = 0

        # Iterate over all states s
        for r in range(GRID_ROWS):
            for c in range(GRID_COLS):
                state = (r, c)

                # Terminal states (Goal and Fire) have fixed utility (their immediate reward)
                # This ensures the policy calculation correctly terminates.
                if state == GOAL_STATE or state == FIRE_STATE:
                    U[r][c] = REWARD(state)
                    PI[r][c] = 'TERMINAL'
                    continue

                # Blocked state utility is non-penalizing directly, and the policy
                # is just to note it's blocked, as no actions are taken from it.
                if state == BLOCKED_STATE:
                    U[r][c] = 0.0
                    PI[r][c] = 'BLOCKED'
                    continue

                # --- Core MDP Value Iteration Update ---

                # Calculate the utility of taking each action 'a' in state 's'
                action_utilities = []
                for action in ACTIONS:
                    # Find the successor state s'
                    s_prime = TRANSITION_MODEL(state, action)

                    # T(s'|s, a) is deterministic (1.0), so the sum simplifies:
                    # Expected Utility = 1.0 * U_prev(s')
                    expected_utility_a = 1.0 * U_prev[s_prime[0]][s_prime[1]]
                    action_utilities.append(expected_utility_a)

                # Find the maximum expected utility (the 'max_a' part of the Bellman equation)
                max_expected_utility = max(action_utilities)

                # Bellman Update: U_i+1(s) = R(s) + gamma * max_a(expected_utility_a)
                new_utility = REWARD(state) + GAMMA * max_expected_utility

                # Update utility table
                U[r][c] = new_utility

                # Track the change for convergence check
                change = abs(U[r][c] - U_prev[r][c])
                max_change = max(max_change, change)

                # Update Policy PI(s): determine the action that yielded the max utility
                # This is the function/method to calculate the optimal policy.
                optimal_action_index = action_utilities.index(max_expected_utility)
                PI[r][c] = ACTIONS[optimal_action_index]

        # 2. Check for convergence (stopping condition)
        if max_change < CONVERGENCE_THRESHOLD:
            break

    # Return the converged Utility values and the Optimal Policy
    return U, PI

# --- AGENT ACTION FUNCTION (Determines what action to take) ---

def DETERMINE_ACTION(state, optimal_policy):
    """
    Function/method to determine what action to take.
    The decision is based upon the pre-calculated Optimal Policy (solution of MDP).
    """
    (r, c) = state

    if state == GOAL_STATE:
        return "GOAL ACHIEVED"
    if state == FIRE_STATE:
        return "AVOIDED FIRE / TERMINATED"
    if state == BLOCKED_STATE:
        return "IMPASSABLE BLOCK"

    # Follow the optimal action derived from Value Iteration
    return optimal_policy[r][c]

# --- GOAL TEST FUNCTION ---

def TEST_GOAL(state):
    """
    Function/method to test if the desired goal is achieved or not.
    """
    return state == GOAL_STATE

# --- SIMULATION (Testing the MDP Solution) ---
# (Uncomment RUN_SIMULATION() at the bottom to execute this logic)

def RUN_SIMULATION():
    print("--- 1. Calculating Optimal Utility and Policy (Solving the MDP) ---")

    Optimal_Utility, Optimal_Policy = CALCULATE_OPTIMAL_POLICY()

    print("\n--- 2. Results (Optimal Policy: Action to take in each state) ---")
    print("Coordinates (1,3) is top-left, (3,1) is bottom-left.")
    # Map to user's 3x3 grid:
    # (1,3) BLOCKED | (2,3) | (3,3) GOAL
    # (1,2)         | (2,2) | (3,2)
    # (1,1) START   | (2,1) | (3,1) FIRE

    # Reverse the rows for printing to match the typical grid orientation (bottom-up)

    # Print Optimal Policy (PI)
    policy_output = []
    policy_output.append(["\nOptimal Policy (Derived from MDP Solution):\n"])

    # Row 0 (User's Row 3: Fire, Middle, Goal)
    policy_output.append([f"({2},{0}): {Optimal_Policy[2][0]}", f"({2},{1}): {Optimal_Policy[2][1]}", f"({2},{2}): {Optimal_Policy[2][2]}"])

    # Row 1 (User's Row 2: Middle, Middle, Middle)
    policy_output.append([f"({1},{0}): {Optimal_Policy[1][0]}", f"({1},{1}): {Optimal_Policy[1][1]}", f"({1},{2}): {Optimal_Policy[1][2]}"])

    # Row 2 (User's Row 1: Start, Middle, Blocked)
    policy_output.append([f"({0},{0}): {Optimal_Policy[0][0]}", f"({0},{1}): {Optimal_Policy[0][1]}", f"({0},{2}): {Optimal_Policy[0][2]}"])

    for row in policy_output:
        print(row)


    print("\n--- 3. Simulation Example from START_STATE (0, 0) ---")
    current_state = START_STATE
    steps = 0

    while not TEST_GOAL(current_state) and current_state != FIRE_STATE and steps < 10:
        action = DETERMINE_ACTION(current_state, Optimal_Policy)

        if action in ['TERMINAL', 'BLOCKED', 'GOAL ACHIEVED', 'AVOIDED FIRE / TERMINATED']:
            print(f"Simulation terminated at state {current_state} with status: {action}")
            break

        next_state = TRANSITION_MODEL(current_state, action)
        # Reward is collected for being in the current state *before* the action
        reward = REWARD(current_state)

        print(f"Step {steps+1}: State {current_state} -> Action {action} -> Next State {next_state} (Reward: {reward:.2f})")

        current_state = next_state
        steps += 1

    print(f"\nFinal Check: Goal Achieved: {TEST_GOAL(current_state)}")

# Execute the simulation
# RUN_SIMULATION()